# Calibrated Data Check Observer's Notebook

This notebook performs various checks on the UHF data that has been flagged and calibrated by the museek pipeline.

# The link for GoogleForms:
https://docs.google.com/forms/d/e/1FAIpQLSc3rmH_jAhCU2TEk9_eYmmC44LiMA27DcG_X7NtrDcf0CB7EA/viewform

## 1. Imports and plotting setup

In [ ]:
import gc
import pickle
import random
from pathlib import Path

import healpy as hp
import numpy as np
import pysm3
from astropy import units as u
from astropy.coordinates import SkyCoord
from matplotlib import pyplot as plt
from matplotlib import rcParams
from scipy.stats import spearmanr
from sklearn.linear_model import HuberRegressor

import museek.util.tools as tl
from museek.enums.result_enum import ResultEnum

plt.style.use("classic")

mpl_params = {
    "font.family": "DejaVu Serif",
    "font.serif": "Times New Roman",
    "font.style": "normal",
    "font.weight": "normal",
    "xtick.major.pad": 4,
    "ytick.major.pad": 2,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
}
rcParams.update(mpl_params)

## 2. Parameters

In [ ]:
# DEFAULT PARAMETERS -- DO NOT REMOVE --
# This cell has "parameters" tag for papermill to recognize the parameters that
# will be overwritten at runtime. Parameters are defined here with their default values.
# When executing the notebook with papermill, a new cell will be injected below this
# cell with the desired parameters being passed to the papermill command, overwritting
# parameter values in this cell. These default parameters can also be inspected by
# calling `papermill --help-notebook <input-notebook>`

block_name: str = "1778715926"  # Block name (CBID)
patch: str = "box14"  # Patch name, e.g. "box14"
base_context_folder: str = "/idia/projects/meerklass/MEERKLASS-1/museek/latest_runs"  # Base context folder. Notebook will look for the data in base_context_folder/patch/block_name/context

## 3. Load pickle once

In [ ]:
context_dir = Path(base_context_folder) / patch / block_name / "context"

with open(context_dir / "aoflagger_plugin_postcalibration.pickle", "rb") as file:
    data_read = pickle.load(file)

calibrated_vis = data_read.get(ResultEnum.CALIBRATED_VIS).result / 1e6
freq = data_read.get(ResultEnum.FREQ_SELECT).result / 1e6
scan_data = data_read.get(ResultEnum.SCAN_DATA).result
r_vis_synch_ant = data_read.get(ResultEnum.CORRELATION_COEFFICIENT_VIS_SYNCH_ANT).result

del data_read
gc.collect()

freq_all = scan_data.frequencies.squeeze / 1e6
timestamps = scan_data.timestamps.array.squeeze()
ra = scan_data.right_ascension.array.squeeze()
dec = scan_data.declination.array.squeeze()

receiver_list = [str(receiver) for receiver in scan_data.receivers]
antenna_list = scan_data._antenna_name_list
receivers = scan_data.receivers
antennas = scan_data.antennas
antenna_names = [ant.name for ant in antennas]

nd_flags = scan_data.flags.array[1] > 0
flags_before_aoflagger = np.sum(scan_data.flags.array[0:4], axis=0) > 0
flags_combine = scan_data.flags.combine(threshold=1.0)
flags = scan_data.flags

freqlow_index = np.argmin(np.abs(freq_all - 580.0))
freqhigh_index = np.argmin(np.abs(freq_all - 1015.0))

visibility_base = np.ma.masked_array(
    scan_data.visibility.array[:, freqlow_index:freqhigh_index, :],
    mask=flags_combine.array[:, freqlow_index:freqhigh_index, :],
    dtype="float32",
)

flags_before_aoflagger = flags_before_aoflagger[:, freqlow_index:freqhigh_index, :]
flags_combine = flags_combine.array[:, freqlow_index:freqhigh_index, :]
flags = flags.array[:, freqlow_index:freqhigh_index, :]

del scan_data
gc.collect()

## 4. Apply extra downlink masking once

In [ ]:
# Work on copies so the base products remain reproducible
visibility = np.ma.masked_array(
    visibility_base.data.copy(),
    mask=np.array(visibility_base.mask, copy=True),
    dtype="float32",
)
calibrated_vis = np.ma.masked_array(
    calibrated_vis.data.copy(),
    mask=np.array(calibrated_vis.mask, copy=True),
)

downlink_ranges = {
    "vodacom": (765, 775),
    "mtn": (801, 811),
    "telkom": (811, 821),
}

freq_ranges = []
for low_freq, high_freq in downlink_ranges.values():
    low_idx = np.argmin(np.abs(freq - low_freq))
    high_idx = np.argmin(np.abs(freq - high_freq))
    freq_ranges.append((low_idx, high_idx))

for low_idx, high_idx in freq_ranges:
    visibility.mask[:, low_idx:high_idx, :] = True
    calibrated_vis.mask[:, low_idx:high_idx, :] = True
    flags_before_aoflagger[:, low_idx:high_idx, :] = True
    flags_combine[:, low_idx:high_idx, :] = True
    flags[:, low_idx:high_idx, :] = True

## 5. Point-source selection

In [ ]:
ra_min = np.nanmin(ra)
ra_max = np.nanmax(ra)
dec_min = np.nanmin(dec)
dec_max = np.nanmax(dec)

ra_pad = 0.05 * (ra_max - ra_min)
dec_pad = 0.05 * (dec_max - dec_min)

right_ascension_median = 0.5 * (ra_min + ra_max)
declination_median = 0.5 * (dec_min + dec_max)

point_source_file_path = (
    "/idia/projects/meerklass/MEERKLASS-1/museek/radio_source_catalog/"
)
point_sources_match_raregion = 0.5 * (ra_max - ra_min)
point_sources_match_decregion = 0.5 * (dec_max - dec_min)

ra_point_source_1jy, dec_point_source_1jy, _ = tl.point_sources_coordinate(
    point_source_file_path,
    right_ascension_median,
    declination_median,
    1.0,
    point_sources_match_raregion,
    point_sources_match_decregion,
)

ra_point_source_5jy, dec_point_source_5jy, _ = tl.point_sources_coordinate(
    point_source_file_path,
    right_ascension_median,
    declination_median,
    5.0,
    point_sources_match_raregion,
    point_sources_match_decregion,
)

print("RA range:", ra_min, ra_max)
print("Dec range:", dec_min, dec_max)
print("N point sources > 1 Jy:", len(ra_point_source_1jy))
print("N point sources > 5 Jy:", len(ra_point_source_5jy))

## 6. Visibility 

In [ ]:
# Keep each masking state explicit. Do not overwrite visibility again below.
visibility_before_aoflagger = np.ma.masked_array(
    visibility.data.copy(),
    mask=np.array(flags_before_aoflagger, copy=True),
)

visibility_after_combine = np.ma.masked_array(
    visibility.data.copy(),
    mask=np.array(flags_combine, copy=True),
)

calibrated_vis_timemedian = np.ma.median(calibrated_vis, axis=0)
calibrated_vis_freqmedian = np.ma.median(calibrated_vis, axis=1)

visibility_timemedian = np.ma.median(visibility, axis=0)
visibility_freqmedian = np.ma.median(visibility, axis=1)

vis_before_aoflagger_timemedian = np.ma.median(visibility_before_aoflagger, axis=0)
vis_before_aoflagger_freqmedian = np.ma.median(visibility_before_aoflagger, axis=1)

visibility_after_combine_timemedian = np.ma.median(visibility_after_combine, axis=0)
visibility_after_combine_freqmedian = np.ma.median(visibility_after_combine, axis=1)

## 7. Flag summary

In [ ]:
flags_antennas = []
for antenna in antenna_list:
    indices = [idx for idx, receiver in enumerate(receiver_list) if antenna in receiver]
    selected_mask = [flags[:, :, :, i] for i in indices]
    flags_antennas.append(np.sum(selected_mask, axis=0))

flags_antennas = np.array(flags_antennas) > 0

flags_SARAO = np.sum(flags_antennas[:, 0:1, :, :], axis=1) > 0
Usable_antennas_SARAO = [not flags_SARAO[i].all() for i in range(len(antennas))]

flags_beforeaoflagger = np.sum(flags_antennas[:, 0:4, :, :], axis=1) > 0
Usable_antennas_beforeaoflagger = [
    not flags_beforeaoflagger[i].all() for i in range(len(antennas))
]

flags_afteraoflagger = np.sum(flags_antennas[:, 0:6, :, :], axis=1) > 0
Usable_antennas_afteraoflagger = [
    not flags_afteraoflagger[i].all() for i in range(len(antennas))
]

flags_afterantennaflagger = np.sum(flags_antennas[:, :, :, :], axis=1) > 0
Usable_antennas_afterantennaflagger = [
    not flags_afterantennaflagger[i].all() for i in range(len(antennas))
]

Usable_antennas_final = [
    not calibrated_vis.mask[:, :, i].all() for i in range(len(antennas))
]

expected_antennas = [f"m{i:03d}" for i in range(64)]
not_in_analysis = [ant for ant in expected_antennas if ant not in antenna_names]
masked_in_final = [
    antenna_names[i] for i, ok in enumerate(Usable_antennas_final) if not ok
]

print("Antennas not included in the analysis:")
print(not_in_analysis)

print("\nAntennas fully masked in final calibrated_vis:")
print(masked_in_final)

print("\nUsable Antennas SARAO:", np.sum(Usable_antennas_SARAO))
print("Usable Antennas before AOflagger:", np.sum(Usable_antennas_beforeaoflagger))
print("Usable Antennas after AOflagger:", np.sum(Usable_antennas_afteraoflagger))
print(
    "Usable Antennas after Antenna flagger:",
    np.sum(Usable_antennas_afterantennaflagger),
)
print("Usable Antennas Final:", np.sum(Usable_antennas_final))

## 8. Flag fraction and correlation overview

In [ ]:
flag_fraction = [
    np.sum(calibrated_vis.mask[:, :, i_ant] > 0)
    / (calibrated_vis.shape[0] * calibrated_vis.shape[1])
    for i_ant in range(len(antennas))
]

x_full = np.arange(len(expected_antennas))

flag_fraction_dict = {
    antenna_names[i]: flag_fraction[i] for i in range(len(antenna_names))
}
corr_dict = {antenna_names[i]: r_vis_synch_ant[i] for i in range(len(antenna_names))}

flag_fraction_full = np.full(len(expected_antennas), np.nan)
corr_full = np.full(len(expected_antennas), np.nan)

for i, ant in enumerate(expected_antennas):
    if ant in flag_fraction_dict:
        flag_fraction_full[i] = flag_fraction_dict[ant]
    if ant in corr_dict:
        corr_full[i] = corr_dict[ant]

masked_idx = [i for i, ant in enumerate(expected_antennas) if ant in masked_in_final]
not_in_analysis_idx = [
    i for i, ant in enumerate(expected_antennas) if ant in not_in_analysis
]

flag_fraction_valid = np.array(
    [flag_fraction_dict[ant] for ant in expected_antennas if ant in flag_fraction_dict]
)
corr_valid = np.array([corr_dict[ant] for ant in expected_antennas if ant in corr_dict])

flag_mean = np.nanmean(flag_fraction_valid)
flag_std = np.nanstd(flag_fraction_valid)
corr_mean = np.nanmean(corr_valid)
corr_std = np.nanstd(corr_valid)

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=False, figsize=(16, 8))

ax1.scatter(x_full, flag_fraction_full, s=14)
ax2.scatter(x_full, corr_full, s=14)

ax1.scatter(
    masked_idx,
    [flag_fraction_full[i] for i in masked_idx],
    color="red",
    s=22,
    zorder=4,
    label="Fully masked in final data",
)
ax2.scatter(masked_idx, [corr_full[i] for i in masked_idx], color="red", s=22, zorder=4)

ax1.scatter(
    not_in_analysis_idx,
    np.zeros(len(not_in_analysis_idx)),
    color="blue",
    marker="x",
    s=60,
    linewidths=1.5,
    zorder=5,
    clip_on=False,
    label="Not included in analysis",
)
ax2.scatter(
    not_in_analysis_idx,
    np.zeros(len(not_in_analysis_idx)),
    color="blue",
    marker="x",
    s=60,
    linewidths=1.5,
    zorder=5,
    clip_on=False,
)

ax1.axhline(flag_mean, linestyle="--", linewidth=1.2, alpha=0.9, label="Mean")
ax2.axhline(corr_mean, linestyle="--", linewidth=1.2, alpha=0.9)

ax1.axhspan(
    max(0.0, flag_mean - flag_std),
    flag_mean + flag_std,
    alpha=0.12,
    label=r"Mean $\pm 1\sigma$",
)
ax2.axhspan(max(0.0, corr_mean - corr_std), corr_mean + corr_std, alpha=0.12)

ax1.set_ylabel("Flagged Fraction", fontsize=18)
ax2.set_ylabel("Correlation Coefficient", fontsize=18)
ax2.set_xlabel("Antenna", fontsize=18)

ax1.set_xlim(-1, 63)
ax2.set_xlim(-1, 63)
ax1.set_ylim(0.0, max(1.05, np.nanmax(flag_fraction_full) + 0.05))
ax2.set_ylim(0.0, max(1.05, np.nanmax(corr_full) + 0.05))

ax1.grid(True, which="major", linestyle=":", linewidth=0.7, alpha=0.7)
ax2.grid(True, which="major", linestyle=":", linewidth=0.7, alpha=0.7)

ax1.set_xticks(x_full)
ax1.set_xticklabels(expected_antennas, rotation=60, ha="center", fontsize=10)
ax2.set_xticks(x_full)
ax2.set_xticklabels(expected_antennas, rotation=60, ha="center", fontsize=10)

ax1.set_title(block_name)

handles, labels = ax1.get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=4,
    fontsize=10,
    frameon=True,
)

fig.align_ylabels()
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
plt.close(fig)

print("Antennas not included in the analysis:")
print(not_in_analysis)

print("\nAntennas fully masked in final calibrated_vis:")
print(masked_in_final)


print("\nUsable Antennas SARAO:", np.sum(Usable_antennas_SARAO))
print("Usable Antennas before AOflagger:", np.sum(Usable_antennas_beforeaoflagger))
print("Usable Antennas after AOflagger:", np.sum(Usable_antennas_afteraoflagger))
print(
    "Usable Antennas after Antenna flagger:",
    np.sum(Usable_antennas_afterantennaflagger),
)
print("Usable Antennas Final:", np.sum(Usable_antennas_final))

print("Flagged Fraction Mean:", f"{flag_mean:.3f}")
print("Flagged Fraction Std:", f"{flag_std:.3f}")
print("Correlation Coefficient Mean:", f"{corr_mean:.3f}")
print("Correlation Coefficient Std:", f"{corr_std:.3f}")

## 9. Non-linearity contamination

Evaluate correlation between stripy pattern and the GSM RFI to quantify non-linearty contamination.
Caveat: works better if the sky model used match the sych emission in the patch

In [ ]:
def ch_finder(frequency_in_MHz, freq):
    f1 = frequency_in_MHz
    ch_1 = np.where(freq < f1)[0][-1]
    ch_2 = np.where(freq > f1)[0][0]
    print("ch:", ch_1, ch_2)
    print("freqs_MHz:", freq[ch_1], freq[ch_2])
    if abs(freq[ch_1] - f1) <= abs(freq[ch_2] - f1):
        return ch_1
    else:
        return ch_2


def zebra_correlation_test(
    vis, freqs, freq_plot, sky_model, nd_times, ch_GSM1, ch_GSM2, receiver_list
):
    gsm = np.sum(vis[np.logical_not(nd_times), ch_GSM1:ch_GSM2, :].data, axis=1)

    spearman_skysub = np.zeros((len(receiver_list), 2), dtype=float)

    ch_plot = ch_finder(freq_plot, freqs)

    for i_rec in range(len(receiver_list)):
        if i_rec % 2 == 0:
            c = SkyCoord(
                ra=ra[:, int((i_rec) / 2)] * u.degree,
                dec=dec[:, int((i_rec) / 2)] * u.degree,
                frame="icrs",
            )
            theta = 90.0 - (c.galactic.b / u.degree).value
            phi = (c.galactic.l / u.degree).value
            synch_I = hp.pixelfunc.get_interp_val(
                sky_model, theta / 180.0 * np.pi, phi / 180.0 * np.pi
            )
            synch_I = synch_I / 10**6.0

        y = vis[:, ch_plot, i_rec]

        # skysub
        huber = HuberRegressor(
            epsilon=1.35,
            max_iter=100,
            alpha=0.0001,
            warm_start=False,
            fit_intercept=True,
            tol=1e-05,
        ).fit(synch_I[np.logical_not(nd_times), None], y[np.logical_not(nd_times)])
        m = huber.coef_
        _ = huber.intercept_

        ysub = (
            y[np.logical_not(nd_times)] - synch_I[np.logical_not(nd_times)] * m
        )  # subtracting only synch not the rest
        spearman_skysub[i_rec, :] = spearmanr(gsm[:, i_rec], ysub)

    return spearman_skysub


# create synchrotron reference
nside = 128  # resolution parameter at which the synchrotron model is to be calculated
beamsize = 57.5  # the beam fwhm used to smooth the Synch model [arcmin]
beam_frequency = 1500.0  # reference frequency at which the beam fwhm are defined [MHz]
# the plot is done at one frequency, this number can be changed as a test
freq_plot = 730.0  #### MHz
sky = pysm3.Sky(nside=nside, preset_strings=["s1"])
map_reference = sky.get_emission(freq_plot * u.MHz).value
map_reference_smoothed = pysm3.apply_smoothing_and_coord_transform(
    map_reference,
    fwhm=beamsize
    * u.arcmin
    * ((beam_frequency * u.MHz) / (freq_plot * u.MHz)).decompose().value,
)
sky_model = map_reference_smoothed[0]

# these are manually defined given the frequancy cut for calibration
freq_cut = freq[freqlow_index:freqhigh_index]
ch_GSM1 = 2600
ch_GSM2 = 2878

# only noise diode off data
nd_times = nd_flags[:, ch_finder(freq_plot, freq_cut), 0]

# zebra estimate
spearman_skysub = zebra_correlation_test(
    visibility,
    freq_cut,
    freq_plot,
    sky_model,
    nd_times,
    ch_GSM1,
    ch_GSM2,
    receiver_list,
)


plt.figure(figsize=(15, 4))
plt.title(block_name)
plt.plot(antenna_list, spearman_skysub[0::2, 0], "bx", label="h")
plt.plot(antenna_list, spearman_skysub[1::2, 0], "gx", label="v")
plt.plot(
    antenna_list, spearman_skysub[:, 0].reshape(-1, 2).mean(axis=1), "r.", label="mean"
)

for ii, ant in enumerate(antenna_list):
    plt.vlines(ii, 0, 1, linestyle=":", color="gray", linewidth=1)
plt.xlim(-0.5, len(antenna_list) - 0.5)
plt.xticks(np.arange(len(antenna_list)), antenna_list, rotation="vertical")
plt.ylim(0, 1)
plt.legend()
plt.show()

spearman_h = spearman_skysub[0::2, 0]
spearman_v = spearman_skysub[1::2, 0]
spearman_mean = spearman_skysub[:, 0].reshape(-1, 2).mean(axis=1)

print(f"H    : mean = {np.nanmean(spearman_h):.3f}, std = {np.nanstd(spearman_h):.3f}")
print(f"V    : mean = {np.nanmean(spearman_v):.3f}, std = {np.nanstd(spearman_v):.3f}")
print(
    f"Mean : mean = {np.nanmean(spearman_mean):.3f}, std = {np.nanstd(spearman_mean):.3f}"
)

plt.figure(figsize=(8, 5))
plt.title(f"{block_name} - boxplot")
plt.boxplot([spearman_h, spearman_v, spearman_mean], tick_labels=["H", "V", "Mean"])
plt.plot(np.ones(len(spearman_h)) * 1, spearman_h, "bx")
plt.plot(np.ones(len(spearman_v)) * 2, spearman_v, "gx")
plt.plot(np.ones(len(spearman_mean)) * 3, spearman_mean, "r.")
plt.ylim(0, 1)
plt.ylabel("Spearman correlation")
plt.show()

## 10. Raw vs calibrated: freq median
Raw vs. Calibrated Data

Examine the frequency median of the raw and calibrated data for all antennas to assess data quality and evaluate the effectiveness of RFI flagging after calibration and following post-calibration aoflagger processing.

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, sharex=True, figsize=(12, 12))

for i_receiver, receiver in enumerate(receiver_list):
    if "h" in str(receiver_list[i_receiver]):
        ax1.plot(freq, visibility_timemedian[:, i_receiver])
    elif "v" in str(receiver_list[i_receiver]):
        ax2.plot(freq, visibility_timemedian[:, i_receiver])

for i_antenna, antenna in enumerate(antenna_list):
    ax3.plot(freq, calibrated_vis_timemedian[:, i_antenna])

ax1.set_ylabel("Raw Vis HH", fontsize=18)
ax2.set_ylabel("Raw Vis VV", fontsize=18)
ax3.set_xlabel("Frequency [MHz]", fontsize=18)
ax3.set_xlim(freq.min() - 5, freq.max() + 5)
ax3.set_ylabel(r"Temperature [$K_{RJ}$]", fontsize=18)
ax1.set_title(block_name + " HH")
ax2.set_title(block_name + " VV")
ax3.set_title(block_name + " Calibrated Vis")

fig.align_ylabels()
plt.show()
plt.close(fig)

## 11. Before AOFlagger spectra and time series -->  Visibility Spectra

Examine the spectra of the raw visibilities before applying AOFlagger, and compare them with the data after flagging. 
This allows us to assess the data quality and determine how much data has been flagged by SARAO or AOFlagger.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(12, 8))

for i_receiver, receiver in enumerate(receiver_list):
    if "h" in str(receiver_list[i_receiver]):
        ax1.plot(freq, vis_before_aoflagger_timemedian[:, i_receiver])
    elif "v" in str(receiver_list[i_receiver]):
        ax2.plot(freq, vis_before_aoflagger_timemedian[:, i_receiver])

ax1.set_ylabel("Vis before AOFlagger HH", fontsize=18)
ax2.set_ylabel("Vis before AOFlagger VV", fontsize=18)
ax2.set_xlabel("Frequency [MHz]", fontsize=18)
ax2.set_xlim(freq.min() - 5, freq.max() + 5)
ax1.set_title(block_name + " timemedian before AOFlagger HH")
ax2.set_title(block_name + " timemedian before AOFlagger VV")
fig.align_ylabels()
plt.show()
plt.close(fig)

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(12, 8))

for i_receiver, receiver in enumerate(receiver_list):
    if "h" in str(receiver_list[i_receiver]):
        ax1.plot(
            (timestamps - timestamps.min()) / 60,
            vis_before_aoflagger_freqmedian[:, i_receiver],
        )
    elif "v" in str(receiver_list[i_receiver]):
        ax2.plot(
            (timestamps - timestamps.min()) / 60,
            vis_before_aoflagger_freqmedian[:, i_receiver],
        )

ax1.set_ylabel("Vis before AOFlagger HH", fontsize=18)
ax2.set_ylabel("Vis before AOFlagger VV", fontsize=18)
ax2.set_xlabel("Time [min]", fontsize=18)
ax2.set_xlim(0, (timestamps.max() - timestamps.min()) / 60)
ax1.set_title(block_name + " freqmedian before AOFlagger HH")
ax2.set_title(block_name + " freqmedian before AOFlagger VV")
fig.align_ylabels()
plt.show()
plt.close(fig)

## 12. Raw time series for random antennas  -->Raw Data

Examine the frequency median of raw data (with frequency average removed) 
from several randomly selected antennas to assess consistency across antennas.

In [ ]:
freq_plot_list = np.arange(600, 1020, 100)

for freq_plot in freq_plot_list:
    index_freq_plot = np.argmin(np.abs(freq - freq_plot))
    fig = plt.figure(figsize=(20, 4))

    ax = fig.add_subplot(1, 2, 1)
    ax2 = fig.add_subplot(1, 2, 2)

    selected_antennas = random.sample(list(antennas), 15)
    i_receiver_list = []
    for ant in selected_antennas:
        i_receiver_list.extend(
            [
                i
                for i, receiver in enumerate(receivers)
                if receiver.antenna_name == ant.name
            ]
        )

    for i_receiver in i_receiver_list:
        visibility_plot = visibility_after_combine[
            :, index_freq_plot, i_receiver
        ] - np.ma.mean(visibility_after_combine[:, index_freq_plot, i_receiver])

        if "h" in str(receiver_list[i_receiver]):
            ax.plot(
                (timestamps - timestamps.min()) / 60,
                visibility_plot,
                label=receiver_list[i_receiver],
            )
        elif "v" in str(receiver_list[i_receiver]):
            ax2.plot(
                (timestamps - timestamps.min()) / 60,
                visibility_plot,
                label=receiver_list[i_receiver],
            )

    ax.set_xlabel("Time [min]", fontsize=18)
    ax.set_ylabel("Raw Vis - average", fontsize=18)
    ax.set_xlim(0, (timestamps.max() - timestamps.min()) / 60)
    ax.set_title(block_name + " hh " + str(freq_plot) + " MHz")
    ax.grid(True, linestyle=":", alpha=0.6)

    ax2.set_xlabel("Time [min]", fontsize=18)
    ax2.set_ylabel("Raw Vis - average", fontsize=18)
    ax2.set_xlim(0, (timestamps.max() - timestamps.min()) / 60)
    ax2.set_title(block_name + " vv " + str(freq_plot) + " MHz")
    ax2.grid(True, linestyle=":", alpha=0.6)

    ax.legend(
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),
        ncol=1,
        fontsize=8,
        frameon=True,
        borderaxespad=0.0,
    )
    ax2.legend(
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),
        ncol=1,
        fontsize=8,
        frameon=True,
        borderaxespad=0.0,
    )

    plt.tight_layout(rect=[0, 0, 0.82, 1])
    plt.show()
    plt.close(fig)

## 13. Median Frequency Raw vis --> Raw maps before AOFlagger (H and V, 1 band --> freq median excluding known rfis) 

 This code reads the 1 Jy and 5 Jy radio source catalogues and selects
 only the sources located inside the RA/Dec region covered by the current
 MeerKAT block. The region is defined around the median pointing position
 of the observation.

 For each antenna, the code produces two panels, corresponding to the
 horizontal and vertical receiver/polarisation channels. The scatter plots
 show the sky pointing positions in RA and Dec, coloured by the median
 visibility before the AOFlagger step.

In [ ]:
from pathlib import Path

import numpy as np

title_fs = 16  # título geral: BOX | BLOCK | ANTENNA | freq
panel_title_fs = 12  # títulos dos painéis
label_fs = 13  # RA / DEC / colorbar labels
tick_fs = 10  # números dos eixos
cbar_tick_fs = 10  # números da colorbar

# =========================================================
# Point-source selection region
# =========================================================
right_ascension_median = np.median(ra)
declination_median = np.median(dec)

point_source_file_path = Path(
    "/idia/projects/meerklass/MEERKLASS-1/museek/radio_source_catalog/"
)

point_sources_match_raregion = 30.0
point_sources_match_decregion = 10.0

file_1jy = point_source_file_path / "1jy_scat-V8_new.txt"
file_5jy = point_source_file_path / "5jy_scat-V0_new.txt"

freq_mean = np.nanmean(freq)


# =========================================================
# Read 1 Jy catalogue
# This file has a header:
# #SOURCE_ID|RA|DEC|NVSS_FLUX|...
# =========================================================
with open(file_1jy) as f:
    header_1jy = f.readline().strip().lstrip("#").split("|")

cat_1jy = np.genfromtxt(
    file_1jy,
    delimiter="|",
    names=header_1jy,
    skip_header=1,
    dtype=None,
    encoding=None,
    autostrip=True,
)

ra_cat_1jy = np.asarray(cat_1jy["RA"], dtype=float)
dec_cat_1jy = np.asarray(cat_1jy["DEC"], dtype=float)


# =========================================================
# Read 5 Jy catalogue
# This file has no header.
# Columns are:
# SOURCE_ID | RA | DEC | FLUX | FREQ | SPEC_INDEX | TYPE
# =========================================================
cat_5jy = np.genfromtxt(
    file_5jy,
    delimiter="|",
    usecols=(1, 2),
    dtype=float,
)

cat_5jy = np.atleast_2d(cat_5jy)

ra_cat_5jy = cat_5jy[:, 0]
dec_cat_5jy = cat_5jy[:, 1]


# =========================================================
# Select 1 Jy sources inside the same RA/Dec region
# =========================================================
delta_ra_1jy = (ra_cat_1jy - right_ascension_median + 180.0) % 360.0 - 180.0
delta_dec_1jy = dec_cat_1jy - declination_median

mask_1jy = (
    (np.abs(delta_ra_1jy) <= point_sources_match_raregion)
    & (np.abs(delta_dec_1jy) <= point_sources_match_decregion)
    & np.isfinite(ra_cat_1jy)
    & np.isfinite(dec_cat_1jy)
)

ra_point_source_1jy = ra_cat_1jy[mask_1jy]
dec_point_source_1jy = dec_cat_1jy[mask_1jy]


# =========================================================
# Select 5 Jy sources inside the same RA/Dec region
# =========================================================
delta_ra_5jy = (ra_cat_5jy - right_ascension_median + 180.0) % 360.0 - 180.0
delta_dec_5jy = dec_cat_5jy - declination_median

mask_5jy = (
    (np.abs(delta_ra_5jy) <= point_sources_match_raregion)
    & (np.abs(delta_dec_5jy) <= point_sources_match_decregion)
    & np.isfinite(ra_cat_5jy)
    & np.isfinite(dec_cat_5jy)
)

ra_point_source_5jy = ra_cat_5jy[mask_5jy]
dec_point_source_5jy = dec_cat_5jy[mask_5jy]


print("Reading 1 Jy:", file_1jy)
print("Reading 5 Jy:", file_5jy)
print("N 1 Jy sources inside region:", len(ra_point_source_1jy))
print("N 5 Jy sources inside region:", len(ra_point_source_5jy))
print("Mean frequency:", freq_mean, "MHz")


# =========================================================
# Plot visibility + 1 Jy and 5 Jy point sources
# =========================================================
for i_antenna, ant in enumerate(antennas):
    fig = plt.figure(figsize=(12, 3))
    ax = fig.add_subplot(1, 2, 1)
    ax2 = fig.add_subplot(1, 2, 2)

    # fig.suptitle(
    #    f"BLOCK {block_name} | ANTENNA {ant.name} | ANTENNA INDEX {i_antenna} | mean frequency = {freq_mean:.1f} MHz",
    #    fontsize=15,
    #    y=1.05,
    # )

    i_receiver_list = [
        i for i, receiver in enumerate(receivers) if receiver.antenna_name == ant.name
    ]

    for i_receiver in i_receiver_list:
        if "h" in str(receiver_list[i_receiver]).lower():
            sc_data = ax.scatter(
                ra[:, i_antenna],
                dec[:, i_antenna],
                c=vis_before_aoflagger_freqmedian[:, i_receiver],
                edgecolor="none",
                cmap="jet",
            )

            ax.scatter(
                ra_point_source_1jy,
                dec_point_source_1jy,
                facecolors="none",
                edgecolors="black",
                marker="s",
                s=40,
                label="1 Jy V8",
            )

            ax.scatter(
                ra_point_source_5jy,
                dec_point_source_5jy,
                color="black",
                marker="+",
                s=70,
                label="5 Jy V0",
            )

            ax.set_title(str(receiver_list[i_receiver]))
            cbar_data = plt.colorbar(sc_data, ax=ax)
            cbar_data.set_label("Vis", fontsize=15)
            # ax.legend(fontsize=8)

        elif "v" in str(receiver_list[i_receiver]).lower():
            sc_data = ax2.scatter(
                ra[:, i_antenna],
                dec[:, i_antenna],
                c=vis_before_aoflagger_freqmedian[:, i_receiver],
                edgecolor="none",
                cmap="jet",
            )

            ax2.scatter(
                ra_point_source_1jy,
                dec_point_source_1jy,
                facecolors="none",
                edgecolors="black",
                marker="s",
                s=40,
                label="1 Jy V8",
            )

            ax2.scatter(
                ra_point_source_5jy,
                dec_point_source_5jy,
                color="black",
                marker="+",
                s=70,
                label="5 Jy V0",
            )

            ax2.set_title(str(receiver_list[i_receiver]))
            cbar_data = plt.colorbar(sc_data, ax=ax2)
            cbar_data.set_label("Vis", fontsize=15)
            # ax2.legend(fontsize=8)

    ax.set_xlabel("RA [Deg]", fontsize=15)
    ax.set_ylabel("DEC [Deg]", fontsize=15)
    ax.set_xlim(np.median(ra) - 25, np.median(ra) + 25)
    ax.set_ylim(np.median(dec) - 8, np.median(dec) + 8)

    ax2.set_xlabel("RA [Deg]", fontsize=15)
    ax2.set_ylabel("DEC [Deg]", fontsize=15)
    ax2.set_xlim(np.median(ra) - 25, np.median(ra) + 25)
    ax2.set_ylim(np.median(dec) - 8, np.median(dec) + 8)

    plt.tight_layout()

    fig.text(
        0.5,
        1.03,
        f"BLOCK {block_name} | ANTENNA {ant.name} | mean frequency = {freq_mean:.1f} MHz",
        ha="center",
        va="bottom",
        fontsize=14,
    )
    # plt.tight_layout(rect=[0, 0, 1, 0.92])
    # plt.savefig(figure_path + block_name + "_" + str(ant.name) + "_waterfall_wenkai.png", dpi=200)
    plt.show()
    plt.close(fig)

## 14. Compare calibrated MeerKAT data with a PySM synchrotron model

 Compare calibrated data with a smoothed PySM synchrotron model.
 Both maps are evaluated on the same antenna track and have their median
 removed before comparison. Bright 1 Jy and 5 Jy catalogue sources are
 overplotted, and the Spearman correlation between data and model is shown.

In [ ]:
from pathlib import Path

import numpy as np

##########  compare with synch model (time median removed for both calibrated data and synch model ) ###########

# =========================================================
# Point-source selection region
# =========================================================
right_ascension_median = np.median(ra)
declination_median = np.median(dec)

point_source_file_path = Path(
    "/idia/projects/meerklass/MEERKLASS-1/museek/radio_source_catalog/"
)

point_sources_match_raregion = 30.0
point_sources_match_decregion = 10.0

file_1jy = point_source_file_path / "1jy_scat-V8_new.txt"
file_5jy = point_source_file_path / "5jy_scat-V0_new.txt"


# =========================================================
# Read 1 Jy catalogue
# This file has a header:
# #SOURCE_ID|RA|DEC|NVSS_FLUX|...
# =========================================================
with open(file_1jy) as f:
    header_1jy = f.readline().strip().lstrip("#").split("|")

cat_1jy = np.genfromtxt(
    file_1jy,
    delimiter="|",
    names=header_1jy,
    skip_header=1,
    dtype=None,
    encoding=None,
    autostrip=True,
)

ra_cat_1jy = np.asarray(cat_1jy["RA"], dtype=float)
dec_cat_1jy = np.asarray(cat_1jy["DEC"], dtype=float)


# =========================================================
# Read 5 Jy catalogue
# This file has no header.
# Columns are:
# SOURCE_ID | RA | DEC | FLUX | FREQ | SPEC_INDEX | TYPE
# =========================================================
cat_5jy = np.genfromtxt(
    file_5jy,
    delimiter="|",
    usecols=(1, 2),
    dtype=float,
)

cat_5jy = np.atleast_2d(cat_5jy)

ra_cat_5jy = cat_5jy[:, 0]
dec_cat_5jy = cat_5jy[:, 1]


# =========================================================
# Select 1 Jy sources inside the same RA/Dec region
# =========================================================
delta_ra_1jy = (ra_cat_1jy - right_ascension_median + 180.0) % 360.0 - 180.0
delta_dec_1jy = dec_cat_1jy - declination_median

mask_1jy = (
    (np.abs(delta_ra_1jy) <= point_sources_match_raregion)
    & (np.abs(delta_dec_1jy) <= point_sources_match_decregion)
    & np.isfinite(ra_cat_1jy)
    & np.isfinite(dec_cat_1jy)
)

ra_point_source_1jy = ra_cat_1jy[mask_1jy]
dec_point_source_1jy = dec_cat_1jy[mask_1jy]


# =========================================================
# Select 5 Jy sources inside the same RA/Dec region
# =========================================================
delta_ra_5jy = (ra_cat_5jy - right_ascension_median + 180.0) % 360.0 - 180.0
delta_dec_5jy = dec_cat_5jy - declination_median

mask_5jy = (
    (np.abs(delta_ra_5jy) <= point_sources_match_raregion)
    & (np.abs(delta_dec_5jy) <= point_sources_match_decregion)
    & np.isfinite(ra_cat_5jy)
    & np.isfinite(dec_cat_5jy)
)

ra_point_source_5jy = ra_cat_5jy[mask_5jy]
dec_point_source_5jy = dec_cat_5jy[mask_5jy]


print("Reading 1 Jy:", file_1jy)
print("Reading 5 Jy:", file_5jy)
print("N 1 Jy sources inside region:", len(ra_point_source_1jy))
print("N 5 Jy sources inside region:", len(ra_point_source_5jy))


# =========================================================
# Synch model setup
# =========================================================
nside = 128  # resolution parameter at which the synchrotron model is to be calculated
beamsize = 57.5  # the beam fwhm used to smooth the Synch model [arcmin]
beam_frequency = 1500.0  # reference frequency at which the beam fwhm are defined [MHz]

sky = pysm3.Sky(
    nside=nside,
    preset_strings=["s1"],
)

calibrated_vis_freqmedian = np.ma.median(calibrated_vis, axis=1)

freq_plot = 730.0  # MHz
freq_plot_up = 730.0 + 1.5
freq_plot_down = 730.0 - 1.5

freq_index_up = np.argmin(np.abs(freq - freq_plot_up))
freq_index_down = np.argmin(np.abs(freq - freq_plot_down))

freq_index_low = min(freq_index_down, freq_index_up)
freq_index_high = max(freq_index_down, freq_index_up) + 1

calibrated_vis_freqplot = np.ma.median(
    calibrated_vis[:, freq_index_low:freq_index_high, :],
    axis=1,
)

freq_mean = np.nanmean(freq[freq_index_low:freq_index_high])

print("Mean frequency used in plot:", freq_mean, "MHz")


# =========================================================
# Plot calibrated data + synch model + 1 Jy and 5 Jy point sources
# =========================================================
for i_antenna, antenna in enumerate(antenna_list):
    if np.ma.getmaskarray(calibrated_vis_freqmedian[:, i_antenna]).all():
        print(antenna + " is masked")

    else:
        ######### produce synch model at a certain frequency and smooth #########
        map_reference = sky.get_emission(freq_plot * u.MHz).value

        map_reference_smoothed = pysm3.apply_smoothing_and_coord_transform(
            map_reference,
            fwhm=beamsize
            * u.arcmin
            * ((beam_frequency * u.MHz) / (freq_plot * u.MHz)).decompose().value,
        )

        ######### map the smoothed synch model to the same sky covered by ra,dec #########
        c = SkyCoord(
            ra=ra[:, i_antenna] * u.degree,
            dec=dec[:, i_antenna] * u.degree,
            frame="icrs",
        )

        theta = 90.0 - (c.galactic.b / u.degree).value
        phi = (c.galactic.l / u.degree).value

        synch_I = hp.pixelfunc.get_interp_val(
            map_reference_smoothed[0],
            theta / 180.0 * np.pi,
            phi / 180.0 * np.pi,
        )

        synch_I = np.ma.masked_array(
            synch_I,
            mask=calibrated_vis_freqplot[:, i_antenna].mask,
        )

        synch_I = synch_I / 10**6.0

        ######## plot #########
        fig = plt.figure(figsize=(12, 3))
        ax = fig.add_subplot(1, 2, 1)
        ax2 = fig.add_subplot(1, 2, 2)

        calibrated_vis_nomedian = calibrated_vis_freqplot[:, i_antenna] - np.ma.median(
            calibrated_vis_freqplot[:, i_antenna]
        )

        sc_data = ax.scatter(
            ra[:, i_antenna],
            dec[:, i_antenna],
            c=calibrated_vis_nomedian,
            edgecolor="none",
            cmap="jet",
            vmin=-0.5,
            vmax=1.0,
        )

        # 1 Jy catalogue: open black squares
        ax.scatter(
            ra_point_source_1jy,
            dec_point_source_1jy,
            facecolors="none",
            edgecolors="black",
            marker="s",
            s=40,
        )

        # 5 Jy catalogue: black plus signs
        ax.scatter(
            ra_point_source_5jy,
            dec_point_source_5jy,
            color="black",
            marker="+",
            s=70,
        )

        ax.set_xlabel("RA [Deg]", fontsize=15)
        ax.set_ylabel("DEC [Deg]", fontsize=15)
        ax.set_title("Calibrated data")
        ax.set_xlim(np.median(ra) - 25, np.median(ra) + 25)
        ax.set_ylim(np.median(dec) - 8, np.median(dec) + 8)

        cbar_data = plt.colorbar(sc_data, ax=ax)
        cbar_data.set_label(r"Calibrated Data [$K_{RJ}$]", fontsize=15)

        synch_I_nomedian = synch_I - np.ma.median(synch_I)

        valid_corr = (
            ~np.ma.getmaskarray(calibrated_vis_nomedian)
            & ~np.ma.getmaskarray(synch_I_nomedian)
            & np.isfinite(calibrated_vis_nomedian.data)
            & np.isfinite(synch_I_nomedian.data)
        )

        if np.sum(valid_corr) >= 2:
            spearman_corr, spearman_p = spearmanr(
                calibrated_vis_nomedian.data[valid_corr],
                synch_I_nomedian.data[valid_corr],
            )
        else:
            spearman_corr = np.nan

        sc_data = ax2.scatter(
            ra[:, i_antenna],
            dec[:, i_antenna],
            c=synch_I_nomedian,
            edgecolor="none",
            cmap="jet",
            vmin=-0.5,
            vmax=1.0,
        )

        # 1 Jy catalogue: open black squares
        ax2.scatter(
            ra_point_source_1jy,
            dec_point_source_1jy,
            facecolors="none",
            edgecolors="black",
            marker="s",
            s=40,
        )

        # 5 Jy catalogue: black plus signs
        ax2.scatter(
            ra_point_source_5jy,
            dec_point_source_5jy,
            color="black",
            marker="+",
            s=70,
        )

        ax2.set_xlabel("RA [Deg]", fontsize=15)
        ax2.set_ylabel("DEC [Deg]", fontsize=15)
        ax2.set_title("correlation coefficient = " + str(round(spearman_corr, 4)))
        ax2.set_xlim(np.median(ra) - 25, np.median(ra) + 25)
        ax2.set_ylim(np.median(dec) - 8, np.median(dec) + 8)

        cbar_data = plt.colorbar(sc_data, ax=ax2)
        cbar_data.set_label(r"Synch [$K_{RJ}$]", fontsize=15)

        plt.tight_layout()

        fig.text(
            0.5,
            0.995,
            f"BLOCK {block_name} | ANTENNA {antenna} | mean frequency = {freq_mean:.1f} MHz",
            ha="center",
            va="top",
            fontsize=12,
        )

        # plt.savefig(
        #     figure_path + block_name + "_" + antenna + "_" + str(round(freq_mean, 3)) + "MHz.png",
        #     dpi=200,
        #     bbox_inches="tight",
        # )

        plt.show()
        plt.close(fig)